# Photon vs $\pi^0$ showers in a W/Si ECAL: classification, anomaly detection, generation (Keras)

In this assignment you work with **Geant4-simulated showers** in a sampling electromagnetic calorimeter (ECAL) and:

1. Choose a **data representation** for the showers
2. Train a **classifier** separating single photons from $\pi^0 \to \gamma\gamma$
3. Train an **anomaly-detection** algorithm on photons only and test it on $\pi^0$ as anomalies; compare it to the classifier
4. Train a **generator** of photon showers and evaluate it with the provided shower-shape test

**What you need to provide**
- A notebook with the solutions of each of the tasks listed above
- For every training: the **training history** (training and validation loss)
- **ROC curve** of the classifier, **ROC curve** of the anomaly detection compared with the classifier (on the same plot)
- **Shower-shape test** output for the generator (the figure and the printed metrics from `ecal_tools.shower_shape_test`)
- **A written answer (in Markdown cells) to the following questions:**
  - For the classifier:
    - How did you decide to represent the dataset and why?
    - why this architecture?
    - why this loss function?
  - For the anomaly detection:
    - How did you decide to represent the dataset and why?
    - why this architecture?
    - How did you decide the figure of merit for anomaly detection?
  - For the generator:
    - How did you decide to represent the dataset and why?
    - why this architecture?
    
Your text will be processed by an LLM to assess how likely it is that it came from an LLM.

**Rules**
- Use the `split` field of the dataset: train on `train`, tune on `val`, report all results on `test`.
- Do **not** modify the shower-shape test functions in `ecal_tools.py`.

## 0. The detector and the dataset

### What is this about?
When a high-energy photon ($\gamma$) hits a block of dense material, it produces a **shower**: a cascade of lower-energy electrons, positrons and photons that deposits its energy over a few centimetres. An **electromagnetic calorimeter (ECAL)** measures the energy of these showers. Ours is a stack of thin tungsten plates, where the shower develops, interleaved with silicon sensors, which measure the energy deposited.

A neutral pion ($\pi^0$) decays immediately into two photons, $\pi^0 \to \gamma\gamma$. At high energy the two photons are very close to each other, so a $\pi^0$ looks almost like a single photon. Telling the two apart is a classic problem in particle physics.

<img src="https://raw.githubusercontent.com/pierinim/tutorials/master/UZH/2026/tutorials/figs/ecal_setup.png" width="850">

**Detector.** 28 layers, each made of 3.5 mm of tungsten followed by a 0.3 mm silicon sensor. Each sensor is segmented into 32×32 square pads of 5×5 mm² (16×16 cm² in total). A pad of a given layer is a **cell**, so the detector has 28×32×32 = 28,672 cells. Only the energy in the silicon is measured. It is calibrated so that the sum of all cells ≈ the energy of the incoming photon.

**Electronics.** Every cell has some electronic noise (0.1 MIP, where 1 MIP ≈ 6.5 MeV is the signal of a single charged particle crossing the pad). To get rid of noise, only cells with more than 0.5 MIP (≈3.3 MeV) are kept: this is called **zero suppression**.

**Events.** Each event is one particle shot perpendicularly at the ECAL from 1.5 m in front of it, with energy uniformly distributed between 10 and 100 GeV, and impact point within ±5 mm of the centre:
- `ecal_gamma.h5`: single photons (50,000 events)
- `ecal_pi0.h5`: $\pi^0 \to \gamma\gamma$ (50,000 events)

Here is one event of each type at 13 GeV. Left: the energy seen from the front (summed over the 28 layers). Right: seen from the side (summed over `iy`). The photon gives one shower. The $\pi^0$ gives two showers, here well separated; at higher energy they merge.

<img src="https://raw.githubusercontent.com/pierinim/tutorials/master/UZH/2026/tutorials/figs/ecal_events.png" width="700">

### Data format: a list of hits per event
Most of the 28,672 cells of an event are empty: a shower lights up only ≈150–650 of them. Instead of storing 28,672 numbers per event (mostly zeros), the dataset stores, for each event, only the cells above threshold, as a **list of hits**. Each hit is 4 numbers:

| `ix` | `iy` | `iz` | `E` |
|---|---|---|---|
| pad column, 0–31 | pad row, 0–31 | layer, 0–27 (0 = front) | energy in GeV |

so an event is an array of shape `(n_hits, 4)`, and **`n_hits` is different for every event**. For example, an event with 3 hits would be

```
[[16, 15,  0, 0.034],
 [15, 15,  1, 0.015],
 [16, 15,  1, 0.293]]
```

In the file the hits of all events are concatenated one after the other, and `offsets[i]:offsets[i+1]` selects the hits of event `i`. You do not need to handle this yourself: `ECALData.hits(i)` returns the `(n_hits, 4)` array of event `i`. Per event you also get `E_true` (the true energy), the gun position `x0, y0` (mm), `label` (0 = $\gamma$, 1 = $\pi^0$) and `split` (0 = train, 1 = val, 2 = test).

**Two representations.** The same event can be seen as a list of hits or as a dense **array** of shape `(28, 32, 32)` (layer, row, column) with zeros in the empty cells, i.e. a 3D image. The helper module `ecal_tools.py` converts between the two:
- `T.list_to_array(data, idx)`: events `idx` of a dataset (or a list of hit arrays) → array `(N, 28, 32, 32)`
- `T.array_to_list(x)`: array `(N, 28, 32, 32)` → list of `N` hit arrays `(n_hits, 4)`

Zero suppression is applied **in both directions**, and again inside the shower-shape test, so cells below 0.5 MIP never enter the evaluation, whatever your model produces. Use whichever representation you like for your models, but note that **the evaluation of the generator (Part 4) uses the array representation**.

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "tensorflow"
import tensorflow as tf
import keras
from keras import layers
import numpy as np
import h5py
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc

In [ ]:
# Download the data and the helper module (re-run the cell if a download is interrupted)
URL = "https://cernbox.cern.ch/remote.php/dav/public-files/SLQGO1eU2Cya43u"
for f in ["ecal_tools.py", "ecal_gamma.h5", "ecal_pi0.h5"]:
    if not os.path.exists(f) or os.path.getsize(f) == 0:
        os.system(f"wget -q -O {f} {URL}/{f}")
    print(f, os.path.getsize(f) // 1_000_000, "MB")
import ecal_tools as T
gam = T.ECALData("ecal_gamma.h5")
pi0 = T.ECALData("ecal_pi0.h5")
print(f"photons: {len(gam)} events, pi0: {len(pi0)} events")
print("detector / digitisation parameters:", gam.attrs)

In [ ]:
# Look at the raw format: the first hits of one event
h = gam.hits(0)
print("event 0: E_true = %.1f GeV, %d hits, sum of hits = %.1f GeV" % (gam.E_true[0], len(h), h[:, 3].sum()))
print("   ix    iy    iz     E[GeV]")
print(h[:8])

# The same event as a dense array (layer, row, column), and back
x = T.list_to_array(gam, [0])          # shape (1, 28, 32, 32)
print("array shape:", x.shape, "  non-zero cells:", (x > 0).sum())
print("back to a list of hits:", T.array_to_list(x)[0].shape)

In [ ]:
# Basic distributions
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
for d, lab in ((gam, "$\\gamma$"), (pi0, "$\\pi^0$")):
    esum = np.add.reduceat(d.E, d.offsets[:-1])
    ax[0].hist(d.E_true, 50, histtype="step", label=lab)
    ax[1].hist(esum / d.E_true, 100, range=(0.8, 1.1), histtype="step", label=lab)
    ax[2].hist(d.n_hits(), 100, histtype="step", label=lab)
ax[0].set_xlabel("E_true [GeV]"); ax[1].set_xlabel("sum of hit energies / E_true"); ax[2].set_xlabel("hits per event")
for a in ax: a.legend()
plt.tight_layout(); plt.show()

In [ ]:
# Event displays: energy summed over all layers (top) and in layer 5 (bottom), log scale
def show_events(d, idx, title):
    img = T.list_to_array(d, idx)
    fig, ax = plt.subplots(2, len(idx), figsize=(3 * len(idx), 6))
    for k, i in enumerate(idx):
        ax[0, k].imshow(np.log10(img[k].sum(0) + 1e-4), vmin=-3, vmax=1.5)
        ax[1, k].imshow(np.log10(img[k, 5] + 1e-4), vmin=-3, vmax=1.0)
        ax[0, k].set_title(f"{title}, E={d.E_true[i]:.0f} GeV")
        for a in ax[:, k]: a.axis("off")
    plt.tight_layout(); plt.show()
show_events(gam, [0, 1, 2, 3], "$\\gamma$")
show_events(pi0, [0, 1, 2, 3], "$\\pi^0$")

## 1. Data representation

The data come as **lists of hits**. Decide how to present them to your networks. You may use different representations for the different tasks.

Things to keep in mind:
- **Memory.** A dense float32 event (`T.list_to_array`) is 28×32×32×4 B ≈ 115 kB, so all events dense do *not* fit in Colab RAM. Think about crops, `float16`, on-the-fly conversion (e.g. a `keras.utils.PyDataset`) and the number of events you really need.
- **Dynamic range.** Cell energies span more than four orders of magnitude.

✏️ **Explain your choice(s) in a Markdown cell.**

In [ ]:
# your data preparation here

## 2. Photon vs $\pi^0$ classifier

Steps:
- Build and train a classifier on the `train` split of both samples (validate on `val`)
- Plot the training history (loss, accuracy)
- Compute the ROC curve and AUC on the `test` split ($\pi^0$ = positive class)
- Study the performance as a function of `E_true` (e.g. AUC, or $\pi^0$ efficiency at fixed photon efficiency, in energy bins) and explain the trend

✏️ **Answer questions related to this task.**
  - How did you decide to represent the dataset and why?
  - why this architecture?
  - why this loss function?



## 3. Anomaly detection trained on photons only


Steps:
- Build and train an anomaly-detection algorithm on photons (the standard) keeping $\pi^0$ (the anomaly) out of the training dataset
- Plot the training history
- Define an anomaly score. Plot its distribution for test photons and test $\pi^0$
- Plot the ROC curve of the anomaly detection **together with the classifier ROC from Part 2**, on the same test events
- Compare the TPR vs FPR for `E_true` < 50 GeV and `E_true` > 50 GeV, and discuss the difference in performance


✏️ **Answer questions related to this task.**
  - How did you decide to represent the dataset and why?
  - why this architecture?
  - How did you decide the figure of merit for anomaly detection?

## 4. Photon shower generator

Steps:
- Build and train a generative model of **photon** showers (the `train` split of `ecal_gamma.h5`)
- Plot the training history
- Generate at least as many showers as there are photons in the `test` split. Convert them to an array of shape `(N, 28, 32, 32)` in GeV: use `T.list_to_array` if your model produces lists of hits, `T.pad_to_full` if it produces a centred crop
- Run the **provided** test: `T.shower_shape_test(real_test_photons, generated)`. It compares the longitudinal and transverse profiles, the total energy, the number of hits, the shower depth and the shower width, after removing low-energy hits (zero supression, usually applied in real-life problems)
- For reference, also run the test on two halves of the real test sample, to see the size of the statistical fluctuations
- Show a few generated showers next to real ones

In [ ]:
real = T.list_to_array(gam, gam.indices('test'))
# generated = ...
# metrics = T.shower_shape_test(real, generated, label_gen='my generator')

✏️ **Explain why you chose this generator and discuss which features it reproduces well and which it does not.**
  - How did you decide to represent the dataset and why?
  - why this architecture?


## 5. Conclusions
Summarise your results in a few sentences. Which part of the problem was hardest, and why?